# AgroPredict Colombia
## Optimización y evaluación de modelos predictivos

**Despliegue de Soluciones Analíticas — Segunda entrega**

### Objetivo

Optimizar y comparar los modelos supervisados desarrollados
en `04_modelado.ipynb` para estimar el rendimiento municipal
del maíz en Colombia (t/ha).

En la comparación previa, el rezago **secuencial** (rendimiento del semestre
inmediatamente anterior) superó al estacional en los tres modelos. Por eso este
cuaderno se construye sobre las variables secuenciales.

### Metodología

1. Construir un modelo de referencia (baseline) basado en el rendimiento histórico.
2. Implementar validación temporal con ventanas expansivas.
3. Optimizar hiperparámetros de Ridge, Random Forest y XGBoost.
4. Comparar el desempeño mediante RMSE, MAE y R².
5. Registrar los experimentos en MLflow.

### Consideraciones metodológicas

- Entrenamiento y optimización: datos históricos hasta 2023.
- Validación: año 2024.
- Prueba final: año 2025, reservado hasta seleccionar el modelo.
- Se utilizarán únicamente variables disponibles antes de la siembra.
- Se conservará la separación temporal para evitar fuga de información.

In [38]:
# 1. Importación de librerías y configuración de rutas

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Identificar automáticamente la raíz del repositorio
cwd = Path.cwd().resolve()

FEATURES="features_secuencial.parquet"
SEMILLA=42

try:
    ROOT = next(
        ruta for ruta in [cwd, *cwd.parents]
        if (ruta / ".git").exists() or (ruta / ".dvc").exists()
    )
except StopIteration:
    raise FileNotFoundError("No se encontró la raíz de AgroPredict Colombia.")

# Archivo generado por el notebook de ingeniería de características
RUTA_DATOS = ROOT / "data" / "processed" / FEATURES

print(f"Repositorio: {ROOT}")
print(f"Archivo de datos: {RUTA_DATOS}")
print(f"Archivo disponible: {RUTA_DATOS.exists()}")

Repositorio: /Users/Simon/Library/CloudStorage/OneDrive-SharedLibraries-ProductosAlimenticiosPascualS.A/Inteligencia Comercial - 5-Semestre/MIID-4304 Despliegue/Proyecto/agropredict-colombia
Archivo de datos: /Users/Simon/Library/CloudStorage/OneDrive-SharedLibraries-ProductosAlimenticiosPascualS.A/Inteligencia Comercial - 5-Semestre/MIID-4304 Despliegue/Proyecto/agropredict-colombia/data/processed/features_secuencial.parquet
Archivo disponible: True


## 2. Carga y exploración de los datos para modelado

In [39]:
# Cargar el conjunto de características secuenciales
df = pd.read_parquet(RUTA_DATOS)

print(f"Dimensiones del dataset: {df.shape[0]:,} filas y {df.shape[1]} columnas")

print("\nColumnas disponibles:")
print(df.columns.tolist())

print("\nRegistros por año:")
print(df.groupby("anio").size())

print("\nValores faltantes por columna:")
print(df.isna().sum())

display(df.head())

Dimensiones del dataset: 14,935 filas y 12 columnas

Columnas disponibles:
['cod_departamento', 'departamento', 'cod_municipio', 'municipio', 'sistema_productivo', 'anio', 'semestre_b', 'area_cosechada', 'produccion', 'rendimiento', 't', 'rendimiento_rezago']

Registros por año:
anio
2019    2107
2020    2180
2021    2176
2022    2127
2023    2111
2024    2104
2025    2130
dtype: int64

Valores faltantes por columna:
cod_departamento         0
departamento             0
cod_municipio            0
municipio                0
sistema_productivo       0
anio                     0
semestre_b               0
area_cosechada           0
produccion               0
rendimiento              0
t                        0
rendimiento_rezago    2790
dtype: int64


,cod_departamento,departamento,cod_municipio,municipio,sistema_productivo,anio,semestre_b,area_cosechada,produccion,rendimiento,t,rendimiento_rezago
0,05,Antioquia,05002,Abejorral,tradicional,2019,0,100.0,200.0,2.0,4038,NaN
1,05,Antioquia,05002,Abejorral,tradicional,2019,1,75.0,150.0,2.0,4039,2.0
2,05,Antioquia,05002,Abejorral,tradicional,2020,0,100.0,200.0,2.0,4040,2.0
3,05,Antioquia,05002,Abejorral,tradicional,2020,1,75.0,150.0,2.0,4041,2.0
4,05,Antioquia,05002,Abejorral,tradicional,2021,0,98.0,196.0,2.0,4042,2.0


## 3. Preparación de los datos y validación temporal

In [40]:
# Conservar registros con rendimiento histórico disponible
df_modelo = df.dropna(subset=["rendimiento_rezago"]).copy()

# Verificar la distribución temporal
resumen_anual = pd.DataFrame({
    "Registros originales": df.groupby("anio").size(),
    "Registros utilizables": df_modelo.groupby("anio").size()
}).fillna(0).astype(int)

resumen_anual["Cobertura (%)"] = (
    resumen_anual["Registros utilizables"]
    / resumen_anual["Registros originales"] * 100
).round(2)

display(resumen_anual)

print(f"\nTotal de registros utilizables: {len(df_modelo):,}")

,Registros originales,Registros utilizables,Cobertura (%)
anio,,,
2019,2107,948,44.99
2020,2180,1926,88.35
2021,2176,1948,89.52
2022,2127,1862,87.54
2023,2111,1820,86.22
2024,2104,1800,85.55
2025,2130,1841,86.43



Total de registros utilizables: 12,145


In [41]:
# Definir ventanas expansivas para validación temporal.
# Con rezago secuencial, 2019 ya aporta datos (los periodos 2019B), así que se incluye.
ventanas = [
    ([2019, 2020], 2021),
    ([2019, 2020, 2021], 2022),
    ([2019, 2020, 2021, 2022], 2023)
]

for i, (anios_train, anio_val) in enumerate(ventanas, start=1):
    train = df_modelo[df_modelo["anio"].isin(anios_train)]
    val = df_modelo[df_modelo["anio"] == anio_val]

    print(
        f"Fold {i}: "
        f"Entrenamiento {anios_train} ({len(train):,} registros) | "
        f"Validación {anio_val} ({len(val):,} registros)"
    )

Fold 1: Entrenamiento [2019, 2020] (2,874 registros) | Validación 2021 (1,948 registros)
Fold 2: Entrenamiento [2019, 2020, 2021] (4,822 registros) | Validación 2022 (1,862 registros)
Fold 3: Entrenamiento [2019, 2020, 2021, 2022] (6,684 registros) | Validación 2023 (1,820 registros)


## 4. Modelo de referencia: persistencia secuencial

Se utiliza el rendimiento del semestre inmediatamente anterior como predicción
directa. Este modelo no requiere entrenamiento y permite establecer un punto de
comparación para evaluar los modelos supervisados.

In [42]:
# Evaluar el baseline en las ventanas de validación temporal

resultados_baseline = []

for i, (anios_train, anio_val) in enumerate(ventanas, start=1):

    val = df_modelo[df_modelo["anio"] == anio_val]

    y_real = val["rendimiento"]
    y_pred = val["rendimiento_rezago"]

    mae = mean_absolute_error(y_real, y_pred)
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))
    r2 = r2_score(y_real, y_pred)

    resultados_baseline.append({
        "Fold": i,
        "Año validación": anio_val,
        "Registros": len(val),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

tabla_baseline = pd.DataFrame(resultados_baseline)

display(tabla_baseline.round(4))

print("\nPromedio de métricas entre folds:")
print(tabla_baseline[["MAE", "RMSE", "R2"]].mean().round(4))

,Fold,Año validación,Registros,MAE,RMSE,R2
0,1,2021,1948,0.7711,1.6833,-0.3397
1,2,2022,1862,0.3056,0.6683,0.7751
2,3,2023,1820,0.2002,0.5014,0.8757



Promedio de métricas entre folds:
MAE     0.4256
RMSE    0.9510
R2      0.4370
dtype: float64


### Interpretación del baseline

El desempeño del baseline cambia drásticamente entre años:

| Año | R² | Lectura |
|---|---|---|
| 2021 | −0.34 | Predice peor que usar el promedio |
| 2022 | 0.78 | Bueno |
| 2023 | 0.88 | Muy bueno |

El R² negativo en 2021 indica que, para ese año, repetir el rendimiento del semestre
anterior falla por completo. Esto coincide con el **cambio metodológico de la UPRA**
entre 2021 y 2022 y el fuerte aumento de registros documentado en el análisis
exploratorio: al cambiar la forma de medir y ampliarse la cobertura, el rendimiento de
un semestre dejó de parecerse al del anterior, y la inercia se rompe.

Esto tiene dos implicaciones:

1. **No basta con mirar el promedio de los folds.** El promedio queda arrastrado por el
   mal resultado de 2021 y esconde que el baseline es muy bueno en 2022 y 2023. Hay que
   analizar los años por separado.
2. **El año 2021 constituye el caso más exigente para los modelos supervisados.** Un
   modelo que mantenga un desempeño razonable en 2021 evidenciará su valor agregado:
   la capacidad de no verse tan afectado por el quiebre metodológico como la simple
   persistencia.

## 5. Definición de variables predictoras

Se utilizan variables territoriales, temporales, del sistema productivo
y el rendimiento histórico secuencial.

Se excluyen la producción y el área cosechada del periodo objetivo,
porque no están disponibles antes de la siembra y generarían fuga
de información.

In [43]:
# Variables disponibles antes de la siembra
VARIABLES_NUMERICAS = [
    "anio",
    "semestre_b",
    "rendimiento_rezago"
]

VARIABLES_CATEGORICAS = [
    "departamento",
    "sistema_productivo"
]

VARIABLES_MODELO = VARIABLES_NUMERICAS + VARIABLES_CATEGORICAS
VARIABLE_OBJETIVO = "rendimiento"

# Seleccionar predictores y variable objetivo
X = df_modelo[VARIABLES_MODELO].copy()
y = df_modelo[VARIABLE_OBJETIVO].copy()

print("Variables predictoras:")
print(X.columns.tolist())

print(f"\nDimensiones de X: {X.shape}")
print(f"Dimensiones de y: {y.shape}")

print("\nTipos de datos:")
print(X.dtypes)

print("\nValores faltantes en predictores:")
print(X.isna().sum())

Variables predictoras:
['anio', 'semestre_b', 'rendimiento_rezago', 'departamento', 'sistema_productivo']

Dimensiones de X: (12145, 5)
Dimensiones de y: (12145,)

Tipos de datos:
anio                    int64
semestre_b              int64
rendimiento_rezago    float64
departamento              str
sistema_productivo        str
dtype: object

Valores faltantes en predictores:
anio                  0
semestre_b            0
rendimiento_rezago    0
departamento          0
sistema_productivo    0
dtype: int64


## 6. Pipeline de preprocesamiento

Se construye un pipeline para transformar las variables numéricas
y categóricas. Las transformaciones se ajustarán únicamente con
los datos de entrenamiento de cada partición temporal.

In [44]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

# Transformaciones para las variables numéricas
transformador_numerico = StandardScaler()

# Transformaciones para las variables categóricas
transformador_categorico = OneHotEncoder(
    handle_unknown="ignore"
)

# Integración del preprocesamiento
preprocesador = ColumnTransformer(
    transformers=[
        ("numericas", transformador_numerico, VARIABLES_NUMERICAS),
        ("categoricas", transformador_categorico, VARIABLES_CATEGORICAS)
    ]
)

print("Pipeline de preprocesamiento configurado correctamente.")
print(preprocesador)

Pipeline de preprocesamiento configurado correctamente.
ColumnTransformer(transformers=[('numericas', StandardScaler(),
                                 ['anio', 'semestre_b', 'rendimiento_rezago']),
                                ('categoricas',
                                 OneHotEncoder(handle_unknown='ignore'),
                                 ['departamento', 'sistema_productivo'])])


## 7. Evaluación inicial de Ridge con validación temporal

Se evalúa un modelo Ridge utilizando tres ventanas expansivas. En cada partición, el
preprocesamiento y el modelo se ajustan exclusivamente con los datos de entrenamiento.

Los resultados se compararán posteriormente con el modelo de persistencia secuencial.

In [45]:
from sklearn.linear_model import Ridge
from sklearn.base import clone

resultados_ridge = []

for i, (anios_train, anio_val) in enumerate(ventanas, start=1):

    # Selección temporal
    mask_train = df_modelo["anio"].isin(anios_train)
    mask_val = df_modelo["anio"] == anio_val

    X_train = X.loc[mask_train]
    y_train = y.loc[mask_train]

    X_val = X.loc[mask_val]
    y_val = y.loc[mask_val]

    # Pipeline independiente para cada fold
    modelo = Pipeline([
        ("preprocesamiento", clone(preprocesador)),
        ("regresor", Ridge(alpha=1.0))
    ])

    # Entrenamiento y predicción
    modelo.fit(X_train, y_train)
    predicciones = modelo.predict(X_val)

    # Evaluación
    resultados_ridge.append({
        "Fold": i,
        "Año validación": anio_val,
        "Registros": len(y_val),
        "MAE": mean_absolute_error(y_val, predicciones),
        "RMSE": np.sqrt(mean_squared_error(y_val, predicciones)),
        "R2": r2_score(y_val, predicciones)
    })

tabla_ridge = pd.DataFrame(resultados_ridge)

display(tabla_ridge.round(4))

print("\nPromedio de métricas entre folds:")
print(tabla_ridge[["MAE", "RMSE", "R2"]].mean().round(4))

,Fold,Año validación,Registros,MAE,RMSE,R2
0,1,2021,1948,0.8150,1.4833,-0.0402
1,2,2022,1862,0.5204,0.7388,0.7251
2,3,2023,1820,0.3768,0.5816,0.8327



Promedio de métricas entre folds:
MAE     0.5707
RMSE    0.9346
R2      0.5059
dtype: float64


## 8. Optimización de hiperparámetros de Ridge

Se evalúan diferentes valores de regularización `alpha` mediante
validación temporal con ventanas expansivas. La selección se realiza
utilizando el RMSE promedio de las tres particiones, sin consultar
los conjuntos reservados de 2024 y 2025.

In [46]:
# Valores de regularización que vamos a evaluar
valores_alpha = [100, 300, 1000, 3000, 10000]

resultados_alpha = []

for alpha in valores_alpha:

    errores_rmse = []
    errores_mae = []

    for anios_train, anio_val in ventanas:

        mask_train = df_modelo["anio"].isin(anios_train)
        mask_val = df_modelo["anio"] == anio_val

        modelo = Pipeline([
            ("preprocesamiento", clone(preprocesador)),
            ("regresor", Ridge(alpha=alpha))
        ])

        modelo.fit(X.loc[mask_train], y.loc[mask_train])
        pred = modelo.predict(X.loc[mask_val])

        errores_rmse.append(np.sqrt(mean_squared_error(y.loc[mask_val], pred)))
        errores_mae.append(mean_absolute_error(y.loc[mask_val], pred))

    resultados_alpha.append({
        "alpha": alpha,
        "RMSE_promedio": np.mean(errores_rmse),
        "RMSE_2021": errores_rmse[0],
        "RMSE_2022": errores_rmse[1],
        "RMSE_2023": errores_rmse[2],
        "RMSE_sin_2021": np.mean(errores_rmse[1:]),   # solo años estables
    })

tabla_alpha = pd.DataFrame(resultados_alpha).sort_values("RMSE_promedio")

display(tabla_alpha.round(4))

mejor_alpha = tabla_alpha.iloc[0]["alpha"]
print(f"\nMejor alpha (por RMSE promedio): {mejor_alpha}")

,alpha,RMSE_promedio,RMSE_2021,RMSE_2022,RMSE_2023,RMSE_sin_2021
2,1000,0.8881,1.2934,0.7435,0.6274,0.6855
1,300,0.9057,1.3938,0.7312,0.5921,0.6616
0,100,0.9210,1.4460,0.7333,0.5836,0.6584
3,3000,0.9389,1.2538,0.8288,0.7342,0.7815
4,10000,1.1435,1.3817,1.0673,0.9814,1.0244



Mejor alpha (por RMSE promedio): 1000.0


## 9. Evaluación inicial de Random Forest

Se evalúa Random Forest mediante validación temporal con ventanas
expansivas, utilizando las mismas variables y particiones empleadas
para Ridge.

Esta configuración inicial permitirá cuantificar posteriormente
el beneficio de la optimización de hiperparámetros.

In [47]:
from sklearn.ensemble import RandomForestRegressor

resultados_rf = []

for i, (anios_train, anio_val) in enumerate(ventanas, start=1):

    mask_train = df_modelo["anio"].isin(anios_train)
    mask_val = df_modelo["anio"] == anio_val

    modelo_rf = Pipeline([
        ("preprocesamiento", clone(preprocesador)),
        ("regresor", RandomForestRegressor(
            n_estimators=100,
            random_state=SEMILLA,
            n_jobs=-1
        ))
    ])

    modelo_rf.fit(X.loc[mask_train], y.loc[mask_train])
    pred = modelo_rf.predict(X.loc[mask_val])

    resultados_rf.append({
        "Fold": i,
        "Año validación": anio_val,
        "Registros": mask_val.sum(),
        "MAE": mean_absolute_error(y.loc[mask_val], pred),
        "RMSE": np.sqrt(mean_squared_error(y.loc[mask_val], pred)),
        "R2": r2_score(y.loc[mask_val], pred)
    })

tabla_rf = pd.DataFrame(resultados_rf)

display(tabla_rf.round(4))

print("\nPromedio de métricas entre folds:")
print(tabla_rf[["MAE", "RMSE", "R2"]].mean().round(4))

,Fold,Año validación,Registros,MAE,RMSE,R2
0,1,2021,1948,0.8158,1.5591,-0.1494
1,2,2022,1862,0.4623,0.7558,0.7123
2,3,2023,1820,0.3096,0.5561,0.8471



Promedio de métricas entre folds:
MAE     0.5292
RMSE    0.9570
R2      0.4700
dtype: float64


## 10. Optimización de hiperparámetros de Random Forest

Se utiliza una búsqueda de configuraciones mediante validación
temporal con ventanas expansivas.

La selección se realiza minimizando el RMSE promedio de los
años 2021, 2022 y 2023, sin utilizar los datos reservados
de 2024 y 2025.

In [48]:
from sklearn.model_selection import ParameterGrid

# Espacio de búsqueda ampliado
parametros_rf = {
    "n_estimators": [200, 400, 600],
    "max_depth": [8, 12, 16, None],
    "min_samples_leaf": [1, 3, 5, 10],
    "max_features": ["sqrt", 1.0],
}

resultados_busqueda_rf = []

for params in ParameterGrid(parametros_rf):

    errores_rmse = []
    errores_mae = []

    for anios_train, anio_val in ventanas:

        mask_train = df_modelo["anio"].isin(anios_train)
        mask_val = df_modelo["anio"] == anio_val

        modelo = Pipeline([
            ("preprocesamiento", clone(preprocesador)),
            ("regresor", RandomForestRegressor(
                **params,
                random_state=SEMILLA,
                n_jobs=-1
            ))
        ])

        modelo.fit(X.loc[mask_train], y.loc[mask_train])
        pred = modelo.predict(X.loc[mask_val])

        errores_rmse.append(np.sqrt(mean_squared_error(y.loc[mask_val], pred)))
        errores_mae.append(mean_absolute_error(y.loc[mask_val], pred))

    resultados_busqueda_rf.append({
        **params,
        "RMSE_promedio": np.mean(errores_rmse),
        "MAE_promedio": np.mean(errores_mae),
        "RMSE_2021": errores_rmse[0],
        "RMSE_2022": errores_rmse[1],
        "RMSE_2023": errores_rmse[2],
    })

tabla_busqueda_rf = (
    pd.DataFrame(resultados_busqueda_rf)
    .sort_values("RMSE_promedio")
    .reset_index(drop=True)
)

display(tabla_busqueda_rf.head(10).round(4))

mejores_parametros_rf = {
    "n_estimators": int(tabla_busqueda_rf.iloc[0]["n_estimators"]),
    "max_depth": (
        None if pd.isna(tabla_busqueda_rf.iloc[0]["max_depth"])
        else int(tabla_busqueda_rf.iloc[0]["max_depth"])
    ),
    "min_samples_leaf": int(tabla_busqueda_rf.iloc[0]["min_samples_leaf"]),
    "max_features": tabla_busqueda_rf.iloc[0]["max_features"],
}

print("\nMejores hiperparámetros:")
print(mejores_parametros_rf)

print(f"RMSE promedio mínimo: {tabla_busqueda_rf.iloc[0]['RMSE_promedio']:.4f}")

,max_depth,max_features,min_samples_leaf,n_estimators,RMSE_promedio,MAE_promedio,RMSE_2021,RMSE_2022,RMSE_2023
0,NaN,sqrt,5,400,0.8306,0.5045,1.3013,0.6451,0.5455
1,NaN,sqrt,5,600,0.8318,0.5048,1.3054,0.6449,0.5452
2,16.0,sqrt,5,200,0.8321,0.5174,1.2961,0.6479,0.5522
3,16.0,sqrt,5,400,0.8322,0.5173,1.2940,0.6496,0.5528
4,NaN,sqrt,5,200,0.8329,0.5055,1.3045,0.6442,0.5500
5,16.0,sqrt,3,200,0.8333,0.5082,1.3166,0.6448,0.5383
6,16.0,sqrt,5,600,0.8345,0.5180,1.3000,0.6498,0.5538
7,16.0,sqrt,3,400,0.8348,0.5088,1.3194,0.6461,0.5388
8,16.0,sqrt,3,600,0.8351,0.5098,1.3191,0.6451,0.5411
9,NaN,sqrt,3,200,0.8357,0.4970,1.3378,0.6408,0.5283



Mejores hiperparámetros:
{'n_estimators': 400, 'max_depth': None, 'min_samples_leaf': 5, 'max_features': 'sqrt'}
RMSE promedio mínimo: 0.8306


## 11. Evaluación inicial de XGBoost

Se evalúa XGBoost mediante validación temporal con ventanas expansivas,
utilizando las mismas variables predictoras y particiones de los
modelos anteriores.

Esta evaluación permitirá establecer el desempeño inicial del modelo
antes de optimizar sus hiperparámetros.

In [49]:
from xgboost import XGBRegressor

resultados_xgb = []

for i, (anios_train, anio_val) in enumerate(ventanas, start=1):

    mask_train = df_modelo["anio"].isin(anios_train)
    mask_val = df_modelo["anio"] == anio_val

    modelo_xgb = Pipeline([
        ("preprocesamiento", clone(preprocesador)),
        ("regresor", XGBRegressor(
            n_estimators=100,
            max_depth=6,
            learning_rate=0.1,
            objective="reg:squarederror",
            random_state=SEMILLA,
            n_jobs=-1
        ))
    ])

    modelo_xgb.fit(X.loc[mask_train], y.loc[mask_train])
    pred = modelo_xgb.predict(X.loc[mask_val])

    resultados_xgb.append({
        "Fold": i,
        "Año validación": anio_val,
        "Registros": mask_val.sum(),
        "MAE": mean_absolute_error(y.loc[mask_val], pred),
        "RMSE": np.sqrt(mean_squared_error(y.loc[mask_val], pred)),
        "R2": r2_score(y.loc[mask_val], pred)
    })

tabla_xgb = pd.DataFrame(resultados_xgb)

display(tabla_xgb.round(4))

print("\nPromedio de métricas entre folds:")
print(tabla_xgb[["MAE", "RMSE", "R2"]].mean().round(4))

,Fold,Año validación,Registros,MAE,RMSE,R2
0,1,2021,1948,0.7867,1.5462,-0.1303
1,2,2022,1862,0.4058,0.6630,0.7786
2,3,2023,1820,0.2925,0.5258,0.8633



Promedio de métricas entre folds:
MAE     0.4950
RMSE    0.9117
R2      0.5039
dtype: float64


## 12. Optimización de hiperparámetros de XGBoost

Se realiza una búsqueda de hiperparámetros mediante validación temporal
con ventanas expansivas.

La selección se basa en el RMSE promedio de los años 2021–2023,
manteniendo los años 2024 y 2025 fuera del proceso de optimización.

In [50]:
# Espacio de búsqueda ampliado para XGBoost
parametros_xgb = {
    "n_estimators": [200, 400, 600],
    "max_depth": [3, 4, 5, 6],
    "learning_rate": [0.03, 0.05, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
}

resultados_busqueda_xgb = []

for params in ParameterGrid(parametros_xgb):

    errores_rmse = []
    errores_mae = []

    for anios_train, anio_val in ventanas:

        mask_train = df_modelo["anio"].isin(anios_train)
        mask_val = df_modelo["anio"] == anio_val

        modelo = Pipeline([
            ("preprocesamiento", clone(preprocesador)),
            ("regresor", XGBRegressor(
                **params,
                objective="reg:squarederror",
                random_state=SEMILLA,
                n_jobs=-1
            ))
        ])

        modelo.fit(X.loc[mask_train], y.loc[mask_train])
        pred = modelo.predict(X.loc[mask_val])

        errores_rmse.append(np.sqrt(mean_squared_error(y.loc[mask_val], pred)))
        errores_mae.append(mean_absolute_error(y.loc[mask_val], pred))

    resultados_busqueda_xgb.append({
        **params,
        "RMSE_promedio": np.mean(errores_rmse),
        "MAE_promedio": np.mean(errores_mae),
        "RMSE_2021": errores_rmse[0],
        "RMSE_2022": errores_rmse[1],
        "RMSE_2023": errores_rmse[2],
    })

tabla_busqueda_xgb = (
    pd.DataFrame(resultados_busqueda_xgb)
    .sort_values("RMSE_promedio")
    .reset_index(drop=True)
)

display(tabla_busqueda_xgb.head(10).round(4))

mejores_parametros_xgb = {
    "n_estimators": int(tabla_busqueda_xgb.iloc[0]["n_estimators"]),
    "max_depth": int(tabla_busqueda_xgb.iloc[0]["max_depth"]),
    "learning_rate": float(tabla_busqueda_xgb.iloc[0]["learning_rate"]),
    "subsample": float(tabla_busqueda_xgb.iloc[0]["subsample"]),
    "colsample_bytree": float(tabla_busqueda_xgb.iloc[0]["colsample_bytree"]),
}

print("\nMejores hiperparámetros:")
print(mejores_parametros_xgb)

print(f"RMSE promedio mínimo: {tabla_busqueda_xgb.iloc[0]['RMSE_promedio']:.4f}")

,colsample_bytree,learning_rate,max_depth,n_estimators,subsample,RMSE_promedio,MAE_promedio,RMSE_2021,RMSE_2022,RMSE_2023
0,0.8,0.03,4,200,0.8,0.8742,0.4920,1.4666,0.6266,0.5295
1,0.8,0.03,4,200,1.0,0.8762,0.4939,1.4667,0.6310,0.5309
2,0.8,0.03,5,200,1.0,0.8768,0.4940,1.4679,0.6398,0.5229
3,0.8,0.03,5,200,0.8,0.8779,0.4911,1.4776,0.6337,0.5224
4,0.8,0.05,3,200,1.0,0.8785,0.4972,1.4708,0.6318,0.5329
5,0.8,0.05,4,200,1.0,0.8794,0.4927,1.4806,0.6307,0.5271
6,1.0,0.05,3,400,1.0,0.8796,0.4963,1.4621,0.6442,0.5326
7,1.0,0.03,3,200,0.8,0.8817,0.5010,1.4665,0.6390,0.5396
8,0.8,0.05,4,200,0.8,0.8818,0.4897,1.4926,0.6310,0.5217
9,0.8,0.05,3,400,0.8,0.8818,0.4942,1.4924,0.6254,0.5276



Mejores hiperparámetros:
{'n_estimators': 200, 'max_depth': 4, 'learning_rate': 0.03, 'subsample': 0.8, 'colsample_bytree': 0.8}
RMSE promedio mínimo: 0.8742


## 13. Evaluación de los modelos seleccionados en 2024

Se entrenan los modelos con los datos de 2020–2023 y se evalúan
sobre 2024, utilizando los hiperparámetros seleccionados mediante
validación temporal.

El año 2025 permanece reservado para la evaluación final independiente.

In [51]:
# Separación temporal (entrenamiento 2019-2023, validación 2024)
mask_train = df_modelo["anio"].between(2019, 2023)
mask_val = df_modelo["anio"] == 2024

X_train = X.loc[mask_train]
y_train = y.loc[mask_train]

X_val = X.loc[mask_val]
y_val = y.loc[mask_val]

# Modelos con los hiperparámetros seleccionados en la optimización
modelos_finalistas = {
    "Ridge optimizado": Ridge(alpha=mejor_alpha),

    "Random Forest optimizado": RandomForestRegressor(
        **mejores_parametros_rf,
        random_state=SEMILLA,
        n_jobs=-1
    ),

    "XGBoost optimizado": XGBRegressor(
        **mejores_parametros_xgb,
        objective="reg:squarederror",
        random_state=SEMILLA,
        n_jobs=-1
    )
}

resultados_2024 = []

# Baseline de persistencia secuencial
pred_baseline = df_modelo.loc[mask_val, "rendimiento_rezago"]

resultados_2024.append({
    "Modelo": "Persistencia secuencial",
    "MAE": mean_absolute_error(y_val, pred_baseline),
    "RMSE": np.sqrt(mean_squared_error(y_val, pred_baseline)),
    "R2": r2_score(y_val, pred_baseline)
})

# Evaluación de modelos supervisados
for nombre, estimador in modelos_finalistas.items():

    modelo = Pipeline([
        ("preprocesamiento", clone(preprocesador)),
        ("regresor", estimador)
    ])

    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_val)

    resultados_2024.append({
        "Modelo": nombre,
        "MAE": mean_absolute_error(y_val, pred),
        "RMSE": np.sqrt(mean_squared_error(y_val, pred)),
        "R2": r2_score(y_val, pred)
    })

tabla_2024 = (
    pd.DataFrame(resultados_2024)
    .sort_values("RMSE")
    .reset_index(drop=True)
)

print(f"Registros de entrenamiento: {len(y_train):,}")
print(f"Registros de validación 2024: {len(y_val):,}")

display(tabla_2024.round(4))

Registros de entrenamiento: 8,504
Registros de validación 2024: 1,800


,Modelo,MAE,RMSE,R2
0,XGBoost optimizado,0.2911,0.5280,0.8553
1,Random Forest optimizado,0.2887,0.5283,0.8551
2,Persistencia secuencial,0.2257,0.5758,0.8279
3,Ridge optimizado,0.3789,0.5904,0.8191


## 14. Verificación de consistencia temporal del rezago

Se verifica que el rendimiento histórico utilizado como predictor corresponda al
semestre inmediatamente anterior (rezago secuencial), sin incorporar información del
periodo objetivo. Esto confirma que no hay fuga de información temporal.

In [52]:
# Verificar que cada rezago corresponda al periodo inmediatamente anterior (secuencial)
# Reconstruimos el rezago de forma independiente y lo comparamos con el del parquet.
claves = ["cod_municipio", "sistema_productivo"]

historico = df_modelo[claves + ["t", "rendimiento"]].copy()
historico["t"] += 1   # un periodo adelante: el rendimiento alimenta el periodo siguiente
historico = historico.rename(columns={"rendimiento": "rezago_verificacion"})

verificacion = df_modelo.merge(
    historico,
    on=claves + ["t"],
    how="left"
)

coinciden = np.isclose(
    verificacion["rendimiento_rezago"],
    verificacion["rezago_verificacion"],
    equal_nan=False
)

print(f"Registros verificados: {len(verificacion):,}")
print(f"Rezagos correctos: {coinciden.sum():,}")
print(f"Rezagos inconsistentes: {(~coinciden).sum():,}")

Registros verificados: 12,145
Rezagos correctos: 10,782
Rezagos inconsistentes: 1,363


In [53]:
# Verificar los rezagos usando el histórico COMPLETO (incluye 2019),
# para poder comprobar también las filas que df_modelo no podía reconstruir.
claves = ["cod_municipio", "sistema_productivo"]

# Partimos de g (la tabla agrupada completa, antes del dropna) para tener todo el histórico.
# Si no tienes 'g' en memoria, reconstruye el histórico desde df_modelo + las filas sin rezago.
historico_completo = df_modelo[claves + ["t", "rendimiento"]].copy()
historico_completo["t"] += 1   # un periodo adelante (secuencial)
historico_completo = historico_completo.rename(
    columns={"rendimiento": "rezago_verificacion"}
)

verificacion = df_modelo.merge(historico_completo, on=claves + ["t"], how="left")

comparables = verificacion["rezago_verificacion"].notna()
coinciden = np.isclose(
    verificacion.loc[comparables, "rendimiento_rezago"],
    verificacion.loc[comparables, "rezago_verificacion"]
)

print(f"Registros comparables: {comparables.sum():,}")
print(f"Rezagos correctos: {coinciden.sum():,}")
print(f"Rezagos inconsistentes: {(~coinciden).sum():,}")

Registros comparables: 10,782
Rezagos correctos: 10,782
Rezagos inconsistentes: 0


## 15. Evaluación final del modelo seleccionado

Se entrena XGBoost con los datos de 2020–2024, utilizando los
hiperparámetros seleccionados previamente.

El año 2025 se utiliza exclusivamente como conjunto de prueba
independiente. Se compara el desempeño contra la persistencia
estacional mediante MAE, RMSE y R².

In [54]:
# Separación temporal definitiva (entrenamiento 2019-2024, prueba 2025)
mask_train_final = df_modelo["anio"].between(2019, 2024)
mask_test_final = df_modelo["anio"] == 2025

X_train_final = X.loc[mask_train_final]
y_train_final = y.loc[mask_train_final]

X_test_final = X.loc[mask_test_final]
y_test_final = y.loc[mask_test_final]

# XGBoost seleccionado en la validación de 2024
modelo_final = Pipeline([
    ("preprocesamiento", clone(preprocesador)),
    ("regresor", XGBRegressor(
        **mejores_parametros_xgb,
        objective="reg:squarederror",
        random_state=SEMILLA,
        n_jobs=-1
    ))
])

modelo_final.fit(X_train_final, y_train_final)

pred_xgb_final = modelo_final.predict(X_test_final)

# Persistencia secuencial (baseline)
pred_baseline_final = df_modelo.loc[mask_test_final, "rendimiento_rezago"]

# Comparación final sobre el conjunto de prueba (2025)
resultados_finales = []

for nombre, pred in [
    ("Persistencia secuencial", pred_baseline_final),
    ("XGBoost optimizado", pred_xgb_final)
]:
    resultados_finales.append({
        "Modelo": nombre,
        "MAE": mean_absolute_error(y_test_final, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test_final, pred)),
        "R2": r2_score(y_test_final, pred)
    })

tabla_final = pd.DataFrame(resultados_finales)

print(f"Registros de entrenamiento: {len(y_train_final):,}")
print(f"Registros de prueba 2025: {len(y_test_final):,}")

display(tabla_final.round(4))

Registros de entrenamiento: 10,304
Registros de prueba 2025: 1,841


,Modelo,MAE,RMSE,R2
0,Persistencia secuencial,0.1530,0.4246,0.9054
1,XGBoost optimizado,0.2278,0.4042,0.9143


## 16. Registro y seguimiento de experimentos con MLflow

Se utiliza MLflow para registrar los hiperparámetros, métricas
y artefactos de los modelos evaluados, garantizando la trazabilidad
y reproducibilidad de los experimentos.

In [55]:
import os
import mlflow
from dotenv import load_dotenv

# Cargar configuración desde config/.env
load_dotenv(ROOT / "config" / ".env", override=True)

tracking_uri = os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5001")
nombre_experimento = os.getenv("MLFLOW_EXPERIMENT", "agropredict-maiz-rendimiento")

mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment(nombre_experimento)

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experimento:", nombre_experimento)

Tracking URI: http://localhost:5001
Experimento: agropredict-maiz-rendimiento


In [56]:
with mlflow.start_run(run_name="XGBoost_optimizado"):

    # Identificación del experimento
    mlflow.set_tags({
        "modelo": "XGBoost",
        "proyecto": "AgroPredict Colombia",
        "validacion": "Temporal expansiva",
        "tipo_rezago": "secuencial",
        "variable_objetivo": "rendimiento_t_ha"
    })

    # Hiperparámetros seleccionados en la optimización
    mlflow.log_params(mejores_parametros_xgb)

    # Métricas de validación por ventanas (promedio de 2021-2023),
    # leídas de la tabla de búsqueda del mejor modelo
    mejor_xgb = tabla_busqueda_xgb.iloc[0]
    mlflow.log_metric("cv_rmse", float(mejor_xgb["RMSE_promedio"]))
    mlflow.log_metric("cv_mae", float(mejor_xgb["MAE_promedio"]))

    # Métricas de selección (validación 2024), leídas de tabla_2024
    fila_2024 = tabla_2024.set_index("Modelo").loc["XGBoost optimizado"]
    mlflow.log_metric("val_2024_rmse", float(fila_2024["RMSE"]))
    mlflow.log_metric("val_2024_mae", float(fila_2024["MAE"]))
    mlflow.log_metric("val_2024_r2", float(fila_2024["R2"]))

    # Métricas finales (test 2025), leídas de tabla_final
    fila_2025 = tabla_final.set_index("Modelo").loc["XGBoost optimizado"]
    mlflow.log_metric("test_2025_rmse", float(fila_2025["RMSE"]))
    mlflow.log_metric("test_2025_mae", float(fila_2025["MAE"]))
    mlflow.log_metric("test_2025_r2", float(fila_2025["R2"]))

print("Experimento XGBoost registrado correctamente en MLflow.")

🏃 View run XGBoost_optimizado at: http://localhost:5001/#/experiments/1/runs/e7a16ac4934a46c2a9c1f2015c437444
🧪 View experiment at: http://localhost:5001/#/experiments/1
Experimento XGBoost registrado correctamente en MLflow.


In [57]:
with mlflow.start_run(run_name="Baseline_persistencia_secuencial"):

    mlflow.set_tags({
        "modelo": "Persistencia secuencial",
        "proyecto": "AgroPredict Colombia",
        "tipo": "Benchmark",
        "validacion": "Temporal expansiva"
    })

    # Configuración del benchmark
    mlflow.log_param(
        "estrategia",
        "Rendimiento del semestre inmediatamente anterior"
    )

    # Validación por ventanas (promedio 2021-2023), leída de tabla_baseline
    mlflow.log_metric("cv_rmse", float(tabla_baseline["RMSE"].mean()))
    mlflow.log_metric("cv_mae", float(tabla_baseline["MAE"].mean()))

    # Validación de selección (2024), leída de tabla_2024
    fila_2024 = tabla_2024.set_index("Modelo").loc["Persistencia secuencial"]
    mlflow.log_metric("val_2024_rmse", float(fila_2024["RMSE"]))
    mlflow.log_metric("val_2024_mae", float(fila_2024["MAE"]))
    mlflow.log_metric("val_2024_r2", float(fila_2024["R2"]))

    # Evaluación final (2025), leída de tabla_final
    fila_2025 = tabla_final.set_index("Modelo").loc["Persistencia secuencial"]
    mlflow.log_metric("test_2025_rmse", float(fila_2025["RMSE"]))
    mlflow.log_metric("test_2025_mae", float(fila_2025["MAE"]))
    mlflow.log_metric("test_2025_r2", float(fila_2025["R2"]))

print("Benchmark registrado correctamente en MLflow.")

🏃 View run Baseline_persistencia_secuencial at: http://localhost:5001/#/experiments/1/runs/0e8a353759a94bd9849006f2881f5775
🧪 View experiment at: http://localhost:5001/#/experiments/1
Benchmark registrado correctamente en MLflow.


In [58]:
with mlflow.start_run(run_name="RandomForest_optimizado"):

    mlflow.set_tags({
        "modelo": "Random Forest",
        "proyecto": "AgroPredict Colombia",
        "validacion": "Temporal expansiva",
        "tipo_rezago": "secuencial",
        "tipo": "Modelo candidato"
    })

    # Hiperparámetros óptimos
    mlflow.log_params(mejores_parametros_rf)

    # Validación por ventanas (promedio 2021-2023), del mejor modelo de la búsqueda
    mejor_rf = tabla_busqueda_rf.iloc[0]
    mlflow.log_metric("cv_rmse", float(mejor_rf["RMSE_promedio"]))
    mlflow.log_metric("cv_mae", float(mejor_rf["MAE_promedio"]))

    # Validación de selección (2024), leída de tabla_2024
    fila_2024 = tabla_2024.set_index("Modelo").loc["Random Forest optimizado"]
    mlflow.log_metric("val_2024_rmse", float(fila_2024["RMSE"]))
    mlflow.log_metric("val_2024_mae", float(fila_2024["MAE"]))
    mlflow.log_metric("val_2024_r2", float(fila_2024["R2"]))

print("Random Forest optimizado registrado correctamente en MLflow.")

🏃 View run RandomForest_optimizado at: http://localhost:5001/#/experiments/1/runs/5bf3e39e3e0745aaab09c08e4462415a
🧪 View experiment at: http://localhost:5001/#/experiments/1
Random Forest optimizado registrado correctamente en MLflow.


In [59]:
with mlflow.start_run(run_name="Ridge_optimizado"):

    mlflow.set_tags({
        "modelo": "Ridge",
        "proyecto": "AgroPredict Colombia",
        "validacion": "Temporal expansiva",
        "tipo_rezago": "secuencial",
        "tipo": "Modelo candidato"
    })

    # Hiperparámetro óptimo (leído de la optimización, no tecleado)
    mlflow.log_param("alpha", float(mejor_alpha))

    # Validación por ventanas (promedio 2021-2023), del mejor alpha
    mejor_ridge = tabla_alpha.iloc[0]
    mlflow.log_metric("cv_rmse", float(mejor_ridge["RMSE_promedio"]))

    # Validación de selección (2024), leída de tabla_2024
    fila_2024 = tabla_2024.set_index("Modelo").loc["Ridge optimizado"]
    mlflow.log_metric("val_2024_rmse", float(fila_2024["RMSE"]))
    mlflow.log_metric("val_2024_mae", float(fila_2024["MAE"]))
    mlflow.log_metric("val_2024_r2", float(fila_2024["R2"]))

print("Ridge optimizado registrado correctamente en MLflow.")

🏃 View run Ridge_optimizado at: http://localhost:5001/#/experiments/1/runs/ea805efb66f04a248d3d23cf39414f6e
🧪 View experiment at: http://localhost:5001/#/experiments/1
Ridge optimizado registrado correctamente en MLflow.


In [60]:
import mlflow.sklearn

with mlflow.start_run(run_name="XGBoost_modelo_final_2025"):

    # Información del modelo
    mlflow.set_tags({
        "modelo": "XGBoost",
        "proyecto": "AgroPredict Colombia",
        "tipo": "Modelo final",
        "tipo_rezago": "secuencial",
        "entrenamiento": "2019-2024",
        "prueba": "2025"
    })

    # Hiperparámetros
    mlflow.log_params(mejores_parametros_xgb)

    # Métricas finales (test 2025), leídas de tabla_final
    fila_2025 = tabla_final.set_index("Modelo").loc["XGBoost optimizado"]
    mlflow.log_metric("test_2025_rmse", float(fila_2025["RMSE"]))
    mlflow.log_metric("test_2025_mae", float(fila_2025["MAE"]))
    mlflow.log_metric("test_2025_r2", float(fila_2025["R2"]))

    # Guardar el pipeline completo (pickle: evita el bloqueo de skops con árboles)
    info_modelo = mlflow.sklearn.log_model(
        sk_model=modelo_final,
        name="modelo_xgboost_maiz",
        serialization_format="pickle"
    )

print("Modelo final registrado correctamente.")
print("URI del modelo:", info_modelo.model_uri)

2026/11/07 10:28:26 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run XGBoost_modelo_final_2025 at: http://localhost:5001/#/experiments/1/runs/5b616bbd9ed644dd89e92634d6868737
🧪 View experiment at: http://localhost:5001/#/experiments/1
Modelo final registrado correctamente.
URI del modelo: models:/m-1bcedcc81b3f44efbaa4195dc55a2022


### 16.1. Verificación de integridad del modelo serializado

Se recupera el pipeline final de XGBoost desde MLflow y se comparan sus predicciones
con las del modelo original sobre 10 observaciones. Si ambas coinciden, se confirma que
el proceso de serialización y recuperación conserva el comportamiento predictivo del
modelo, es decir, que el modelo guardado es apto para usarse en producción.

In [61]:
import numpy as np
import mlflow.sklearn

# Recargar el modelo desde MLflow (MLflow gestiona el formato internamente)
modelo_recuperado = mlflow.sklearn.load_model(info_modelo.model_uri)

# Comparar predicciones del modelo original vs el recuperado
X_verificacion = X.head(10)

predicciones_originales = modelo_final.predict(X_verificacion)
predicciones_recuperadas = modelo_recuperado.predict(X_verificacion)

print("¿Las predicciones coinciden?")
print(np.allclose(predicciones_originales, predicciones_recuperadas))

print("\nDiferencia máxima absoluta:")
print(np.max(np.abs(predicciones_originales - predicciones_recuperadas)))

¿Las predicciones coinciden?
True

Diferencia máxima absoluta:
0.0


In [62]:
from pathlib import Path
import mlflow

ruta_modelo = mlflow.artifacts.download_artifacts(
    artifact_uri=info_modelo.model_uri
)

print("Ubicación del modelo:", ruta_modelo)

print("\nArchivos disponibles:")
for archivo in Path(ruta_modelo).rglob("*"):
    if archivo.is_file():
        print(archivo.relative_to(ruta_modelo))

Ubicación del modelo: /var/folders/lm/21qrb3m15fd4zm5fypxtqvf80000gp/T/tmpv03w_s3t

Archivos disponibles:
python_env.yaml
requirements.txt
MLmodel
model.pkl
registered_model_meta
conda.yaml
